# 决策树第五课：C4.5 决策树

本课目标：理解 C4.5 为什么要在 ID3 的基础上引入“信息增益率”。

一句话版：**ID3 看信息增益，C4.5 看信息增益率。**

## 1. ID3 哪里不够好？

ID3 每次选择信息增益最大的特征。这个思路很自然，但有一个明显问题：**它偏爱取值特别多的特征。**

比如一个特征叫 `用户编号`：

| 用户编号 | 是否批准贷款 |
| --- | --- |
| U001 | 批准 |
| U002 | 拒绝 |
| U003 | 批准 |
| U004 | 拒绝 |

如果每个人的编号都不一样，那么按 `用户编号` 分裂以后，每个分支都只有 1 个样本。每个分支都会变得非常纯，条件熵接近 0，信息增益会很高。

但这显然不是好规则。因为新用户来了，编号从没见过，树根本不知道该怎么走。

## 2. C4.5 的核心改进

C4.5 不直接用信息增益，而是使用信息增益率：

$$GainRatio(D,A)=\frac{Gain(D,A)}{SplitInfo(D,A)}$$

其中：

- $Gain(D,A)$：这个特征让标签不确定性减少了多少。
- $SplitInfo(D,A)$：这个特征自己把数据切得有多碎。

直觉上：如果一个特征靠“切得很碎”才获得高信息增益，C4.5 会给它一个惩罚。

## 3. 分裂信息 SplitInfo 是什么？

分裂信息只看特征 $A$ 把样本分成了哪些组，以及每组占比是多少。它不看标签。

$$SplitInfo(D,A)=-\sum_{v\in Values(A)}\frac{|D_v|}{|D|}\log_2\frac{|D_v|}{|D|}$$

你可以把它理解成：**这个特征本身的分叉复杂度。**

如果一个特征只有两个取值，而且分组比较集中，SplitInfo 不会太大。

如果一个特征有很多取值，把数据切成很多小碎片，SplitInfo 就会变大。

## 4. 继续用贷款审批例子

| 编号 | 有工作 | 有房产 | 用户编号 | 是否批准贷款 |
| ---: | --- | --- | --- | --- |
| 1 | 是 | 是 | U001 | 批准 |
| 2 | 否 | 是 | U002 | 批准 |
| 3 | 是 | 否 | U003 | 批准 |
| 4 | 是 | 否 | U004 | 批准 |
| 5 | 否 | 否 | U005 | 拒绝 |
| 6 | 否 | 否 | U006 | 拒绝 |
| 7 | 是 | 是 | U007 | 批准 |
| 8 | 否 | 否 | U008 | 拒绝 |

我们比较三个特征：

- `有工作`
- `有房产`
- `用户编号`

`用户编号` 是故意放进来的坏特征，用来观察 ID3 为什么会被取值多的特征吸引。

## 5. 先回顾信息增益

整个数据集里：批准 5 条，拒绝 3 条。

$$H(D)=-(\frac{5}{8}\log_2\frac{5}{8}+\frac{3}{8}\log_2\frac{3}{8})=0.9544$$

上一课算过：

$$Gain(D, 有工作)=0.5488$$

$$Gain(D, 有房产)=0.3476$$

现在看 `用户编号`。

每个用户编号只对应 1 条样本，所以按 `用户编号` 分裂以后，每个分支都只有一个样本。单个样本的熵是 0。

因此：

$$H(D|用户编号)=0$$

$$Gain(D, 用户编号)=H(D)-0=0.9544$$

如果只看信息增益，ID3 会选择 `用户编号`。但这显然很糟糕。

## 6. 手算“有工作”的 SplitInfo

`有工作` 把 8 条数据分成两组：

- 有工作 = 是：4 条，占 $\frac{4}{8}=0.5$
- 有工作 = 否：4 条，占 $\frac{4}{8}=0.5$

所以：

$$SplitInfo(D, 有工作)=-(0.5\log_2 0.5+0.5\log_2 0.5)$$

$$SplitInfo(D, 有工作)=1$$

信息增益率：

$$GainRatio(D, 有工作)=\frac{0.5488}{1}=0.5488$$

## 7. 手算“有房产”的 SplitInfo

`有房产` 把 8 条数据分成两组：

- 有房产 = 是：3 条，占 $\frac{3}{8}=0.375$
- 有房产 = 否：5 条，占 $\frac{5}{8}=0.625$

所以：

$$SplitInfo(D, 有房产)=-(0.375\log_2 0.375+0.625\log_2 0.625)$$

这和总熵数值刚好一样，因为比例也是 3:5：

$$SplitInfo(D, 有房产)=0.9544$$

信息增益率：

$$GainRatio(D, 有房产)=\frac{0.3476}{0.9544}=0.3642$$

## 8. 手算“用户编号”的 SplitInfo

`用户编号` 把 8 条数据分成 8 组，每组 1 条。

每组占比都是：

$$\frac{1}{8}=0.125$$

所以：

$$SplitInfo(D, 用户编号)=-8\times(\frac{1}{8}\log_2\frac{1}{8})$$

$$\log_2\frac{1}{8}=-3$$

$$SplitInfo(D, 用户编号)=-8\times(\frac{1}{8}\times -3)=3$$

虽然它的信息增益很高：

$$Gain(D, 用户编号)=0.9544$$

但它的信息增益率是：

$$GainRatio(D, 用户编号)=\frac{0.9544}{3}=0.3181$$

这样一来，`用户编号` 就没有 `有工作` 好了。

## 9. 对比结果

| 特征 | 信息增益 Gain | 分裂信息 SplitInfo | 信息增益率 GainRatio |
| --- | ---: | ---: | ---: |
| 有工作 | 0.5488 | 1.0000 | 0.5488 |
| 有房产 | 0.3476 | 0.9544 | 0.3642 |
| 用户编号 | 0.9544 | 3.0000 | 0.3181 |

如果用 ID3 的信息增益，会选 `用户编号`。

如果用 C4.5 的信息增益率，会选 `有工作`。

这就是 C4.5 比 ID3 更稳的地方：它不只看“你让标签变纯了多少”，还看“你是不是靠把数据切得太碎才做到的”。

In [ ]:
from collections import Counter, defaultdict
from math import log2

def entropy(values):
    total = len(values)
    counts = Counter(values)
    result = -sum((count / total) * log2(count / total) for count in counts.values())
    return 0.0 if abs(result) < 1e-12 else result

def split_by_feature(rows, feature):
    groups = defaultdict(list)
    for row in rows:
        groups[row[feature]].append(row)
    return groups

def conditional_entropy(rows, feature, target):
    total = len(rows)
    groups = split_by_feature(rows, feature)
    return sum(
        (len(group_rows) / total) * entropy([row[target] for row in group_rows])
        for group_rows in groups.values()
    )

def information_gain(rows, feature, target):
    labels = [row[target] for row in rows]
    return entropy(labels) - conditional_entropy(rows, feature, target)

def split_info(rows, feature):
    total = len(rows)
    groups = split_by_feature(rows, feature)
    result = -sum(
        (len(group_rows) / total) * log2(len(group_rows) / total)
        for group_rows in groups.values()
    )
    return 0.0 if abs(result) < 1e-12 else result

def gain_ratio(rows, feature, target):
    denominator = split_info(rows, feature)
    if denominator == 0:
        return 0.0
    return information_gain(rows, feature, target) / denominator

In [ ]:
loan_data = [
    {'有工作': '是', '有房产': '是', '用户编号': 'U001', '是否批准贷款': '批准'},
    {'有工作': '否', '有房产': '是', '用户编号': 'U002', '是否批准贷款': '批准'},
    {'有工作': '是', '有房产': '否', '用户编号': 'U003', '是否批准贷款': '批准'},
    {'有工作': '是', '有房产': '否', '用户编号': 'U004', '是否批准贷款': '批准'},
    {'有工作': '否', '有房产': '否', '用户编号': 'U005', '是否批准贷款': '拒绝'},
    {'有工作': '否', '有房产': '否', '用户编号': 'U006', '是否批准贷款': '拒绝'},
    {'有工作': '是', '有房产': '是', '用户编号': 'U007', '是否批准贷款': '批准'},
    {'有工作': '否', '有房产': '否', '用户编号': 'U008', '是否批准贷款': '拒绝'},
]

target = '是否批准贷款'
features = ['有工作', '有房产', '用户编号']

print(f"H(D) = {entropy([row[target] for row in loan_data]):.4f}\n")

for feature in features:
    gain = information_gain(loan_data, feature, target)
    split = split_info(loan_data, feature)
    ratio = gain_ratio(loan_data, feature, target)
    print(f"特征：{feature}")
    print(f"  信息增益 Gain = {gain:.4f}")
    print(f"  分裂信息 SplitInfo = {split:.4f}")
    print(f"  信息增益率 GainRatio = {ratio:.4f}\n")

## 10. C4.5 还改进了什么？

除了信息增益率，C4.5 相比 ID3 还有几个常见改进：

- 可以处理连续特征，比如年龄、收入、温度。
- 可以处理缺失值。
- 建树以后可以剪枝，减轻过拟合。
- 生成的规则仍然比较容易解释。

不过这节课先抓住最核心的一点：**C4.5 用信息增益率削弱了 ID3 对多取值特征的偏爱。**

## 11. 本课小结

- ID3 使用信息增益选择特征。
- 信息增益容易偏爱取值很多的特征。
- C4.5 使用信息增益率：

$$GainRatio(D,A)=\frac{Gain(D,A)}{SplitInfo(D,A)}$$

- SplitInfo 衡量特征把数据切得有多碎。
- 如果一个特征靠制造很多小分支获得高信息增益，它的信息增益率会被压低。

记忆方式：**信息增益问“纯了多少”，信息增益率还要追问“你是不是切太碎了”。**